<a href="https://colab.research.google.com/github/auralyawirawan/-DistilBERT-Based-Sentiment-Analysis-on-COVID-19-Tweets/blob/Distilbert-Covid-Sentiment-Analysis/FP_SML_Sub.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

##Part 1

In [ ]:
!pip install transformers torch scikit-learn tqdm --quiet

import library


In [ ]:
import pandas as pd
import torch
from torch.utils.data import Dataset, DataLoader
from transformers import (
    DistilBertTokenizerFast, DistilBertForSequenceClassification
)
from torch.optim import AdamW

from sklearn.metrics import accuracy_score, classification_report, confusion_matrix
from sklearn.model_selection import train_test_split
from tqdm import tqdm
from sklearn.preprocessing import LabelEncoder


In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns


upload data

In [ ]:
from google.colab import files
uploaded = files.upload()

read data

In [ ]:
df = pd.read_csv("Corona.csv", encoding="latin1")
df.head()

##PART 2

data info

In [ ]:
df.info()

data encoding

In [ ]:
le = LabelEncoder()
df["label"] = le.fit_transform(df["Sentiment"])
num_labels = df["label"].nunique()

train_texts, test_texts, train_labels, test_labels = train_test_split(
    df["OriginalTweet"].tolist(),
    df["label"].tolist(),
    test_size=0.2,
    random_state=42,
    stratify=df["label"]
)

In [ ]:
df.head()

load tokenizer

In [ ]:
tokenizer = DistilBertTokenizerFast.from_pretrained("distilbert-base-uncased")

dataset class

In [ ]:
class TweetDataset(Dataset):
    def __init__(self, texts, labels, tokenizer):
        self.enc = tokenizer(
            texts,
            truncation=True,
            padding=True,
            max_length=64,
            return_tensors="pt"
        )
        self.labels = torch.tensor(labels, dtype=torch.long)

    def __len__(self):
        return len(self.labels)

    def __getitem__(self, idx):
        return {
            "input_ids": self.enc["input_ids"][idx],
            "attention_mask": self.enc["attention_mask"][idx],
            "labels": self.labels[idx]
        }

##Part 3

In [ ]:
distil_model = DistilBertForSequenceClassification.from_pretrained(
    "distilbert-base-uncased", num_labels=5
).to(device)

optimizer2 = torch.optim.AdamW(distil_model.parameters(), lr=3e-5)


In [ ]:
# DistilBERT tokenizer
distil_tokenizer = DistilBertTokenizerFast.from_pretrained("distilbert-base-uncased")

# Dataset
distil_train = TweetDataset(train_texts, train_labels, distil_tokenizer)
distil_test  = TweetDataset(test_texts, test_labels, distil_tokenizer)

# DataLoader
train_loader = DataLoader(distil_train, batch_size=8, shuffle=True)
test_loader  = DataLoader(distil_test, batch_size=16)


In [ ]:
loss_history = []

for epoch in range(5):
    distil_model.train()
    total_loss = 0

    for batch in train_loader:
        optimizer2.zero_grad()

        input_ids = batch["input_ids"].to(device)
        attention_mask = batch["attention_mask"].to(device)
        labels = batch["labels"].to(device)

        outputs = distil_model(
            input_ids=input_ids,
            attention_mask=attention_mask,
            labels=labels
        )

        loss = outputs.loss
        total_loss += loss.item()

        loss.backward()
        optimizer2.step()

    avg_loss = total_loss / len(train_loader)
    loss_history.append(avg_loss)

    print(f"[DistilBERT] Epoch {epoch+1} — Loss: {avg_loss:.4f}")



## Part 4

evaluation Metric

In [ ]:
# DISTILBERT CONFUSION MATRIX

cm_distil = confusion_matrix(trues, preds)  # DistilBERT's trues & preds from Part 3

plt.figure(figsize=(8, 6))
sns.heatmap(cm_distil, annot=True, fmt="d", cmap="Greens",
            xticklabels=le.classes_,
            yticklabels=le.classes_)
plt.title("DistilBERT Confusion Matrix")
plt.xlabel("Predicted Label")
plt.ylabel("True Label")
plt.show()


In [ ]:
plt.figure(figsize=(7,5))
plt.plot(range(1, len(loss_history)+1), loss_history, marker='o')
plt.title("DistilBERT Training Loss Curve")
plt.xlabel("Epoch")
plt.ylabel("Loss")
plt.grid()
plt.show()


In [ ]:
import pandas as pd

distil_acc = accuracy_score(trues, preds)

results_df = pd.DataFrame({
    "Model": ["DistilBERT"],
    "Accuracy": [distil_acc]
})

results_df

In [ ]:
distil_model.eval()
preds, trues = [], []

with torch.no_grad():
    for batch in test_loader:
        input_ids = batch["input_ids"].to(device)
        attention_mask = batch["attention_mask"].to(device)

        outputs = distil_model(
            input_ids=input_ids,
            attention_mask=attention_mask
        )

        logits = outputs.logits
        predictions = torch.argmax(logits, dim=1)

        preds.extend(predictions.cpu().numpy())
        trues.extend(batch["labels"].numpy())


print("Accuracy:", accuracy_score(trues, preds))
print("\nClassification Report:")
print(classification_report(trues, preds, target_names=le.classes_))
